============================================
Brasil Olímpico - Scripts do SSMS -> JSON do site
Roda no Google Colab
============================================
Passo 1: No SSMS, clique com botão direito no banco jogosolimpicos
  Tarefas > Gerar Scripts > Tabelas + Dados > Salvar como .sql
Passo 2: No Colab, suba esse .sql (botão de upload no painel esquerdo)
Passo 3: Rode as células abaixo, na ordem
Passo 4: Baixe o dados_brasil_olimpico.json, o index.html E o index_standalone.html atualizados e suba os 3 no GitHub

Por que 3 arquivos? O site tem 2 versões do HTML, do mesmo JSON:
  1) dados_brasil_olimpico.json (usado pela versão leve no GitHub Pages)
  2) index.html — versão LEVE (~324 KB, sem dados embutidos, busca o JSON com fetch). Para GitHub Pages / servidor.
  3) index_standalone.html — versão STANDALONE (1 arquivo só, com os dados embutidos em <script id="embedded-data">). Para dois cliques / WhatsApp / e-mail.
As 2 versões têm exatamente os mesmos dados, gráficos, filtros e visual. A única diferença é onde o dado mora.


In [ ]:
# --- CÉLULA 1: Upload do .sql ---
from google.colab import files
import re, json, collections

print("Selecione o arquivo .sql gerado pelo SSMS")
uploaded = files.upload()  # vai abrir a janela de upload
nome_arquivo = list(uploaded.keys())[0]
print(f"Arquivo recebido: {nome_arquivo}")


In [ ]:
# --- CÉLULA 2: Parse do .sql ---
# O SSMS gera em UTF-16. Tente UTF-16, se falhar, tente UTF-8.
raw = open(nome_arquivo, 'rb').read()
try:
    t = raw.decode('utf-16')
except:
    t = raw.decode('utf-8', errors='ignore')

lines = [l for l in t.splitlines() if l.startswith("INSERT [jogosolimpicos].")]

def parse_values(line):
    m = re.search(r'VALUES \((.*)\)\s*$', line)
    s = m.group(1)
    parts = []; cur = ""; in_str = False; i = 0
    while i < len(s):
        ch = s[i]
        if ch == "'":
            if i+1 < len(s) and s[i+1] == "'":
                cur += "''"; i += 2; continue
            in_str = not in_str; cur += ch; i += 1; continue
        if ch == "," and not in_str:
            parts.append(cur.strip()); cur = ""; i += 1; continue
        cur += ch; i += 1
    parts.append(cur.strip())
    def clean(v):
        v = v.strip()
        if v == "NULL": return None
        mm = re.match(r"CAST\(N'(.*)' AS Date\)", v)
        if mm: return mm.group(1).replace("''", "'")
        if v.startswith("N'") and v.endswith("'"): return v[2:-1].replace("''", "'")
        try: return int(v)
        except: return v
    return [clean(x) for x in parts]

atletas = {}   # id -> dict
edicoes = {}   # id -> dict
medalhas_map = {}  # id -> dict
rel = []       # lista de [ID, MedalhaID, AtletaID, EdicaoID]

for l in lines:
    if ".[atleta]" in l:
        r = parse_values(l)
        # [AtletaID, Nome_Principal, Nome_Complemento, Sexo, Data_Nascimento, Data_Morte, Cidade, Estado, Pais]
        atletas[r[0]] = {"id": r[0], "nome": r[1], "nome_oficial": r[2], "sexo": r[3],
                         "nasc": r[4], "cidade": r[6], "estado": r[7], "pais": r[8]}
    elif ".[edicao]" in l:
        r = parse_values(l)
        # [EdicaoID, Cidade, Pais, Ano, Verao_Inverno, Abertura, Encerramento, Inicio, Fim]
        edicoes[r[0]] = {"id": r[0], "cidade": r[1], "pais": r[2], "ano": r[3], "estacao": r[4]}
    elif ".[medalha]" in l:
        r = parse_values(l)
        # [MedalhaID, EdicaoID, Data, Medalha, Individual_Coletivo, Modalidade, Evento, Genero]
        medalhas_map[r[0]] = {"id": r[0], "edicao_id": r[1], "data": r[2], "tipo": r[3],
                              "indiv": r[4], "modalidade": r[5], "evento": r[6], "genero": r[7]}
    elif ".[medalhascomatletas]" in l:
        rel.append(parse_values(l))

print(f"atletas: {len(atletas)} | edicoes: {len(edicoes)} | medalhas: {len(medalhas_map)} | vínculos: {len(rel)}")


In [ ]:
# --- CÉLULA 3: Validações rápidas ---
# Se alguma falhar, pare aqui e confira o banco antes de gerar o JSON
assert len(atletas) == len(set(atletas.keys())), "ID de atleta duplicado"
assert len(medalhas_map) == len(set(medalhas_map.keys())), "ID de medalha duplicado"
atleta_ids = set(atletas.keys()); edicao_ids = set(edicoes.keys()); medalha_ids = set(medalhas_map.keys())
assert all(m["edicao_id"] in edicao_ids for m in medalhas_map.values()), "Medalha sem edição"
assert all(r[1] in medalha_ids and r[2] in atleta_ids for r in rel), "Vínculo com FK quebrada"
assert len(set((r[1], r[2]) for r in rel)) == len(rel), "Vínculo duplicado (medalha+atleta)"
print("✅ Validações OK - sem FK quebrada, sem duplicado")

# Totais por tipo
from collections import Counter
print("Totais:", Counter(m["tipo"] for m in medalhas_map.values()))


In [ ]:
# --- CÉLULA 4: Montar o JSON do site ---
# Mapa medalha -> atletas
medalha_para_atletas = collections.defaultdict(list)
for r in rel:
    medalha_para_atletas[r[1]].append(atletas[r[2]])

def genero_label(g):
    return {"M": "Masculino", "F": "Feminino", "F&M": "Misto"}.get(g, g)

medalhas = []
for mid, m in sorted(medalhas_map.items()):
    ed = edicoes[m["edicao_id"]]
    ats = medalha_para_atletas[mid]
    det = [{"nome": a["nome"], "nome_oficial": a["nome_oficial"],
            "busca": (a["nome"] + " " + (a["nome_oficial"] or "")).strip()} for a in ats]
    medalhas.append({
        "id": mid, "ano": ed["ano"], "cidade": ed["cidade"], "estacao": ed["estacao"],
        "modalidade": m["modalidade"], "evento": m["evento"], "genero": m["genero"],
        "genero_label": genero_label(m["genero"]), "indiv": m["indiv"], "data": m["data"],
        "medalha_tipo": m["tipo"], "atletas": [a["nome"] for a in ats],
        "atletas_detalhe": det,
        "busca_atletas": " ".join(d["busca"] for d in det)
    })

# by_edicao: TODAS as 55 edições, incluindo zeros
by_edicao = []
for eid, e in sorted(edicoes.items(), key=lambda x: (x[1]["ano"], x[1]["estacao"])):
    c = Counter(m["tipo"] for m in medalhas_map.values() if m["edicao_id"] == eid)
    by_edicao.append({"ano": e["ano"], "cidade": e["cidade"], "estacao": e["estacao"],
                      "ouro": c.get("Ouro", 0), "prata": c.get("Prata", 0), "bronze": c.get("Bronze", 0),
                      "total": sum(c.values())})

# by_modalidade
by_mod = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for m in medalhas_map.values():
    k = m["modalidade"]; by_mod[k]["total"] += 1
    by_mod[k]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_modalidade = [{"modalidade_nome": k, "total_medalhas": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]}
                 for k, v in sorted(by_mod.items(), key=lambda x: -x[1]["total"])]

# by_estado: conta VÍNCULOS medalha + atleta (1 medalha coletiva conta 1x por atleta que ganhou medalha)
by_est = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for r in rel:
    a = atletas[r[2]]; m = medalhas_map[r[1]]
    if a["pais"] != "Brasil": continue
    est = a["estado"] or "Não informado"
    by_est[est]["total"] += 1
    by_est[est]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_estado = [{"estado_nome": k, "total_vinculos": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]}
             for k, v in sorted(by_est.items(), key=lambda x: -x[1]["total"])]

# by_atleta + ranking
by_atl = collections.defaultdict(lambda: {"ouro":0,"prata":0,"bronze":0,"total":0})
for r in rel:
    m = medalhas_map[r[1]]
    by_atl[r[2]]["total"] += 1
    by_atl[r[2]]["ouro" if m["tipo"]=="Ouro" else "prata" if m["tipo"]=="Prata" else "bronze"] += 1
by_atleta = []
ranking = []
for aid, v in sorted(by_atl.items(), key=lambda x: (-x[1]["total"], -x[1]["ouro"], -x[1]["prata"])):
    a = atletas[aid]
    by_atleta.append({"atleta_nome": a["nome"], "total_medalhas": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"]})
    ranking.append({"nome": a["nome"], "sexo": a["sexo"], "estado": a["estado"], "pais": a["pais"], "cidade": a["cidade"],
                    "total": v["total"], "ouro": v["ouro"], "prata": v["prata"], "bronze": v["bronze"],
                    "nome_oficial": a["nome_oficial"], "busca": (a["nome"] + " " + (a["nome_oficial"] or "")).strip()})

# by_genero por edição (com zeros)
by_ge = collections.defaultdict(lambda: {"Masculino":0,"Feminino":0,"Misto":0})
for m in medalhas:
    by_ge[(m["ano"], m["estacao"])][m["genero_label"]] += 1
by_genero_edicao_completa = []
for e in by_edicao:
    v = by_ge.get((e["ano"], e["estacao"]), {"Masculino":0,"Feminino":0,"Misto":0})
    by_genero_edicao_completa.append({"ano": e["ano"], "cidade": e["cidade"], "estacao": e["estacao"],
                                      "Masculino": v["Masculino"], "Feminino": v["Feminino"], "Misto": v["Misto"],
                                      "total": sum(v.values())})
# Compatibilidade: by_genero_ano (agrupado por ano)
by_genero_ano = []
for ano in sorted(set(m["ano"] for m in medalhas)):
    agg = {"Masculino":0,"Feminino":0,"Misto":0}
    for m in medalhas:
        if m["ano"] == ano: agg[m["genero_label"]] += 1
    by_genero_ano.append({"ano": ano, **agg})

# --- idades_detalhe ---
# 1 linha por VÍNCULO medalha + atleta (um time de vôlei com 12 atletas vira 12 linhas)
# idade = data da medalha - data de nascimento (leva em conta ano bissexto)
from datetime import datetime

def calc_idade(nasc_str, medalha_data_str):
    if not nasc_str or not medalha_data_str:
        return None
    try:
        nasc = datetime.strptime(str(nasc_str)[:10], "%Y-%m-%d")
        med = datetime.strptime(str(medalha_data_str)[:10], "%Y-%m-%d")
    except:
        return None
    dias = (med - nasc).days
    if dias < 0:
        return None
    return dias / 365.25

idades_detalhe = []
for r in rel:
    mid, aid = r[1], r[2]
    a = atletas[aid]
    m = medalhas_map[mid]
    ed = edicoes[m["edicao_id"]]
    idade = calc_idade(a["nasc"], m["data"])
    if idade is None:
        continue  # sem data de nascimento -> pula, não quebra o JSON
    nasc_dt = datetime.strptime(str(a["nasc"])[:10], "%Y-%m-%d")
    med_dt = datetime.strptime(str(m["data"])[:10], "%Y-%m-%d")
    anos = med_dt.year - nasc_dt.year - ((med_dt.month, med_dt.day) < (nasc_dt.month, nasc_dt.day))
    try:
        aniv = nasc_dt.replace(year=nasc_dt.year + anos)
    except ValueError:
        aniv = nasc_dt.replace(year=nasc_dt.year + anos, day=28)  # quem nasceu em 29/02
    dias = (med_dt - aniv).days
    idade_texto = f"{anos} anos e {dias} dias"
    idades_detalhe.append({
        "nome": a["nome"],
        "idade": round(idade, 2),
        "idade_texto": idade_texto,
        "ano": ed["ano"],
        "medalha_tipo": m["tipo"],
        "modalidade": m["modalidade"],
        "data_medalha": m["data"],
        "data_nascimento": a["nasc"],
    })

idades = [x["idade"] for x in idades_detalhe]

print(f"idades_detalhe: {len(idades_detalhe)} linhas")
print(f"Sem nascimento (puladas): {len(rel) - len(idades_detalhe)}")
mais_jovem = min(idades_detalhe, key=lambda x: x["idade"])
mais_velho = max(idades_detalhe, key=lambda x: x["idade"])
print(f"Mais jovem: {mais_jovem['nome']} - {mais_jovem['idade_texto']} em {mais_jovem['data_medalha'][:4]}")
print(f"Mais velho: {mais_velho['nome']} - {mais_velho['idade_texto']} em {mais_velho['data_medalha'][:4]}")

dados = {"medalhas": medalhas, "by_edicao": by_edicao, "by_modalidade": by_modalidade,
         "by_estado": by_estado, "by_atleta": by_atleta, "ranking": ranking,
         "by_genero_edicao_completa": by_genero_edicao_completa, "by_genero_ano": by_genero_ano,
         "edicoes": by_edicao, "by_edicao_completa": by_edicao,
         "idades": idades, "idades_detalhe": idades_detalhe}

with open("dados_brasil_olimpico.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, ensure_ascii=False, indent=2)

print(f"✅ Gerado dados_brasil_olimpico.json: {len(medalhas)} medalhas, {len(by_edicao)} edições, {len(ranking)} medalhistas")
# Validação cruzada: 3 agrupamentos diferentes têm que dar o mesmo total
soma_edicao = sum(x["total"] for x in by_edicao)
soma_genero = sum(x["total"] for x in by_genero_edicao_completa)
soma_modalidade = sum(x["total_medalhas"] for x in by_modalidade)
soma_vinculos = sum(x["total"] for x in ranking)
print(f"   Soma por edição: {soma_edicao} (deve ser = {len(medalhas)} medalhas)")
print(f"   Soma por gênero: {soma_genero} (deve ser = {len(medalhas)} medalhas)")
print(f"   Soma por modalidade: {soma_modalidade} (deve ser = {len(medalhas)} medalhas)")
print(f"   Soma vínculos medalha + atleta: {soma_vinculos} (deve ser = {len(rel)} vínculos)")
print(f"   Top atleta: {ranking[0]['nome']} com {ranking[0]['total']} medalhas")


In [ ]:
# --- CÉLULA 5: Gerar index.html (leve) + index_standalone.html (1 arquivo só) do mesmo JSON ---
# NUNCA mais gere a versão híbrida (com os 2 ao mesmo tempo).
# Como funciona:
#   - index.html (LEVE): sem dados embutidos, com o comentário
#     <!-- Dados carregados de dados_brasil_olimpico.json (versão web, sem cópia embutida para ficar leve) -->
#     no lugar da tag. Busca o JSON com fetch(). Só funciona em servidor / GitHub Pages.
#     Se abrir com dois cliques, mostra o aviso amarelo "Não consegui carregar o data.json automaticamente".
#   - index_standalone.html (STANDALONE): mesma base + a tag
#     <script id="embedded-data" type="application/json"> com o JSON minificado no lugar do comentário.
#     Já tem os dados dentro. Não precisa do JSON ao lado. Para dois cliques / WhatsApp / e-mail.
#   - As 2 versões têm exatamente os mesmos dados, gráficos, filtros e visual.
#     A única diferença é onde o dado mora.
# O que esta célula espera:
#   - Você sobe o index.html LEVE atual (o que tem o comentário placeholder).
#   - Se você subir por engano o index_standalone.html (que tem a tag embedded-data),
#     ela tira a tag e recria a leve a partir dele, para não duplicar dados.
print("Selecione o index.html LEVE atual do projeto Brasil Olímpico")
print("(Se você só tem o index_standalone.html, pode subir ele também — a célula tira a tag e recria a leve)")
uploaded_html = files.upload()
nome_html = list(uploaded_html.keys())[0]
print(f"HTML recebido: {nome_html}")

html = open(nome_html, encoding="utf-8").read()
novo_dados = json.dumps(dados, ensure_ascii=False, separators=(",", ":"))
placeholder = "<!-- Dados carregados de dados_brasil_olimpico.json (versão web, sem cópia embutida para ficar leve) -->"
novo_tag = '<script id="embedded-data" type="application/json">\n' + novo_dados + '\n</script>'

# Passo 1: Garantir que estamos partindo da base LEVE (sem embedded-data)
# Caso A: já é a leve (tem o placeholder) — usa como está
if placeholder in html:
    html_leve = html
    print("✅ Achada a base leve (placeholder) — usando como está.")
# Caso B: é a standalone (tem a tag embedded-data) — tira a tag e põe o placeholder no lugar
elif re.search(r'<script id="embedded-data"[^>]*>.*?</script>', html, flags=re.S):
    html_leve, n = re.subn(r'<script id="embedded-data"[^>]*>.*?</script>', placeholder, html, count=1, flags=re.S)
    print(f"✅ Achada a standalone (tem embedded-data) — removida a tag e recriada a leve (n={n}).")
# Caso C: é um HTML muito antigo (ainda usa var Ac = ...) — tira o bloco var Ac e põe o placeholder
# ATENÇÃO: o regex simples var Ac = {...}; corta no primeiro "};", então aqui fazemos a remoção por contagem de chaves.
else:
    print("⚠️ Não achei nem o placeholder nem a tag embedded-data. Tentando remover var Ac antigo por contagem de chaves...")
    start = html.find("var Ac=")
    if start == -1:
        print("❌ Não achei nem placeholder, nem embedded-data, nem var Ac. Confira se o arquivo é o index.html do Brasil Olímpico.")
        html_leve = None
    else:
        i = start + len("var Ac=")
        while i < len(html) and html[i] in " \t\n\r":
            i += 1
        brace_start = i
        depth = 0
        in_str = None
        esc = False
        j = brace_start
        end = None
        while j < len(html):
            ch = html[j]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == in_str:
                    in_str = None
            else:
                if ch == '"' or ch == "'":
                    in_str = ch
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        k = j + 1
                        while k < len(html) and html[k] in " \t\n\r":
                            k += 1
                        end = k + 1 if k < len(html) and html[k] == ";" else j + 1
                        break
            j += 1
        if end is None:
            print("❌ Não consegui achar o fim do var Ac. Confira o arquivo.")
            html_leve = None
        else:
            html_leve = html[:start] + placeholder + html[end:]
            print("✅ Removido var Ac antigo por contagem de chaves — leve recriada.")

if html_leve is not None:
    # Verificação: a base leve tem que ter o upload manual (filePicker) como fallback.
    # Se o template for antigo e não tiver, avisa — não tente inventar o bloco aqui, use o index.html leve atual que já tem.
    if "filePicker" not in html_leve:
        print("⚠️ Aviso: a base leve não tem o campo filePicker (upload manual do JSON).")
        print("   Use como base o index.html leve atual, que já tem o aviso amarelo + <input id=\"filePicker\">.")
        print("   Se continuar assim, a próxima geração vai perder o recurso de upload manual.")
    # Passo 2: A leve é a base como está (já tem o placeholder, sem dados embutidos)
    # Garantir que a leve NÃO tenha tag embedded-data (por segurança, se sobrou alguma)
    if re.search(r'<script id="embedded-data"', html_leve):
        print("⚠️ A leve ainda tinha uma tag embedded-data — removendo por segurança.")
        html_leve, _ = re.subn(r'<script id="embedded-data"[^>]*>.*?</script>', placeholder, html_leve, count=1, flags=re.S)
    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html_leve)
    print(f"✅ Gerado index.html (LEVE, sem dados embutidos) com {len(medalhas)} medalhas no JSON ao lado.")

    # Passo 3: A standalone é a base + a tag embedded-data com o JSON minificado no lugar do placeholder
    if placeholder not in html_leve:
        print("❌ Não achei o placeholder na leve para gerar a standalone. Confira o arquivo.")
        html_standalone = None
    else:
        html_standalone = html_leve.replace(placeholder, novo_tag, count=1)
        with open("index_standalone.html", "w", encoding="utf-8") as f:
            f.write(html_standalone)
        print(f"✅ Gerado index_standalone.html (STANDALONE, 1 arquivo só, com os dados dentro) com {len(medalhas)} medalhas embutidas.")
        print(f"   As 2 versões têm exatamente os mesmos dados, gráficos, filtros e visual.")


In [ ]:
# --- CÉLULA 6: Baixar os 3 arquivos atualizados ---
files.download("dados_brasil_olimpico.json")
files.download("index.html")
files.download("index_standalone.html")
print("Baixe os 3 arquivos e suba os 3 no GitHub no lugar dos antigos. Pronto!")
print("GitHub Pages: use index.html (leve) + dados_brasil_olimpico.json na mesma pasta.")
print("Dois cliques / WhatsApp / e-mail: use index_standalone.html (1 arquivo só, já com os dados dentro).")
